In [ ]:
import pandas as pd
import plotly.express as px

# 학습, 테스트 데이터 분포 확인
validaion set과 test set의 성능 차이가 너무 크게 나타남에 따라, 테스트 데이터의 분포에 drift가 일어났을 가능성이 있기에 이를 확인하기 위한 탐색적 데이터 분석을 수행합니다. id 컬럼이 시간에 따라 증가한다고 가정합니다.

In [ ]:
df = pd.read_csv("../data/exp02/train.csv")
df_test = pd.read_csv("../data/exp02/test.csv")

df.drop("defect", axis=1, inplace=True)
df_test.drop("defect", axis=1, inplace=True)
df["type"] = "train"
df_test["type"] = "test"
df_all = pd.concat([df, df_test])

### 학습 데이터, 테스트 데이터 분포 비고

확인내용
- 두 데이터의 분포가 확연이 차이남을 확인
- 테스트 2에는 제품 2만, 테스트 1에는 제품 1 데이터만 있음을 확인

조치내용
- 단순 제품에 의한 차이인지, 시간 변화에 따른 drift인지 확인

In [ ]:
# 수치형 컬럼 선택 (id, Shot 제외)
numeric_cols = (
    df_all.select_dtypes(include=["int64", "float64"])
    .drop(columns=["id", "Shot"], errors="ignore")
    .columns.tolist()
)
df_melted = df_all.melt(
    id_vars="type", value_vars=numeric_cols, var_name="variable", value_name="value"
)

# plotly로 boxplot 그리기
fig = px.box(
    df_melted,
    x="type",
    y="value",
    facet_col="variable",
    facet_col_wrap=len(numeric_cols) // 5 + 1,
    color="type",
    boxmode="group",
    height=1200,
    width=1600,
    title="Boxplots of Variables by Defect",
)

fig.update_layout(showlegend=True)
fig.update_yaxes(matches=None, showticklabels=True)
fig.show()

### 학습 데이터, 테스트 데이터 분포 비고(시계열)

확인내용
- 시간에 따라 구분해서 봤을 때, 제품에 따라 분포가 변하는 것을 확인할 수 있음
- 단 velocity의 경우, 제품1에서 제품 2로 넘어가는 시점에 갑자기 속도가 증가하는 경향이 있음
- 몇몇 변수는 시간에 따라 점차 증가함(Shot, Coolant_Temp)
- 몇몇 변수는 데이터 분포가 크게 변화함(Spray_Time, Spray_1_Time, Spray_2_Time, Cylinder_Pressure, Coolant_Pressure, Air_Pressure, Casting_Pressure)

조치내용
- 제품에 따라 대부분의 변수 분포가 변하므로, 각 제품별로 다른 모델을 만드는 것이 합리적임
- 분포가 크게 변하는 변수는 분석에서 제외(noise가 될 수 있음)
- 제품 타입을 변경하기 전, Velocity(금형에 용탕을 투입하는 속도)가 증가하는 경향이 있음을 전달.(제품 타입 변경 전, 생산량을 기한 내에 맞추기 위한 행위일 수 있음). 또한 이 때 불량이 많이 발생할 수 있음을 전달

In [ ]:
numeric_cols = (
    df_all.select_dtypes(include=["int64", "float64"])
    .drop(columns=["id"], errors="ignore")
    .columns.tolist()
)
df_melted = df_all.melt(
    id_vars=["id", "Product_Type"],
    value_vars=numeric_cols,
    var_name="variable",
    value_name="value",
)
df_melted = df_melted.sort_values(by=["id", "Product_Type"])

# Create line plots showing the distribution
fig = px.line(
    df_melted,
    x="id",
    y="value",
    color="Product_Type",
    facet_col="variable",
    facet_col_wrap=2,
    title="Time Series View of Numeric Features",
    template="plotly_white",
)
fig.update_layout(showlegend=True, height=1200)
fig.update_yaxes(matches=None, showticklabels=True)
fig.show()